# 14 · Retrieval reliability analysis

Retrieval reliability sprint: where retrieval fails on the real corpus (corpus_v2), how much document fragmentation matters, whether a small evidence-sufficiency layer (neighbouring chunks + one bounded targeted search) makes the evidence more complete, what the source-quality checks find, and how all of this compares with the B0/E1/E2 baselines and the completed reranking experiments - with accuracy *and* cost. Local, isolated indexes only; frozen benchmark versions are unchanged; draft labels are marked as such.

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

In [ ]:
import re, csv, collections
from datetime import date
from scipy.stats import norm
from jsonl_io import open_jsonl
REL = A.DATA / "experiments/reliability"; CQ = A.DATA / "experiments/chunking"; FIG = A.RESEARCH / "figures/reliability"
REP = A.RESEARCH / "reports/reliability"; REP.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(A.RESEARCH.parent / "backend"))
from app.services.assistant.evidence import required_information
from app.services.source_quality import SourceQualityConfig, assess_documents
GB, EMP = "d60bc363-0aae-44e5-b0c3-df8c61ad6f66", "96b0fee4-18e9-4ea2-adb5-db654c9ac238"
REC = {}
for ds in ("pilot_v0.1", "heldout_v0.2"):
    REC.update({r["question_id"]: dict(r, dataset=ds) for r in json.loads((A.DATA / f"benchmarks/retrieval_{ds}.json").read_text())["records"]})
DRAFT = {r["question_id"]: r for r in map(json.loads, (A.DATA / "review/passage_labels_v0.3_draft.jsonl").open(encoding="utf-8"))}
FROZEN = {q: set(r["expected_document_ids"]) for q, r in REC.items()}
AI = {q: FROZEN[q] | {d["document_id"] for d in DRAFT[q]["proposed_additional_documents"]} for q in REC}
MAN = {i["id"]: i for i in json.loads((A.DATA / "snapshots/corpus_v2/corpus_manifest.json").read_text())["items"]}
CORPUS = {d["id"]: d for d in json.loads((CQ / "C0_existing/corpus.json").read_text())}
TITLE = {k: v["title"] for k, v in MAN.items()}
TR = {}
for snap in ("corpus_v2", "corpus_v2_p1"):
    f = REL / snap / "traces.jsonl"
    if f.exists() or f.with_name(f.name + ".gz").exists():
        with open_jsonl(f) as fh:
            TR[snap] = pd.DataFrame([json.loads(x) for x in fh])
print({k: (len(v), sorted(v.pipeline.unique())) for k, v in TR.items()})
print({k: json.loads((REL / k / "run.json").read_text())["unmatched_candidate_passages"] for k in TR})

def supports(q, doc, p, labels=AI):
    """Chunking-independent passage support (v0.3-draft criteria, pending review)."""
    if doc not in labels[q]:
        return False
    ps = DRAFT[q]["passage_support"]
    text = (p["text"] or "").lower()
    return any(t.lower() in text for t in ps["evidence_terms"]) or (doc == GB and p.get("page") in set(ps["guidebook_pages"]))

def has_criteria(q):
    ps = DRAFT[q]["passage_support"]; return bool(ps["evidence_terms"] or ps["guidebook_pages"])

STALE = re.compile(r"((?:19|20)\d{2})\s*[.년]\s*(\d{1,2})\s*[.월]\s*(\d{1,2})\s*\.?\s*(?:일)?\s*\)?\s*(?:까지|until)")
def stale_statement(text):
    for y, m, d in STALE.findall(text or ""):
        try:
            if date(int(y), int(m), int(d)) < date.today(): return f"{y}.{m}.{d} 까지"
        except ValueError: pass
    return None

## 1 · Failure audit (corpus_v2, the assistant's current evidence path E2S)

Every question is checked against eight failure types (several can apply). The rules are automatic and deliberately simple; the examples below each type are actual traces from the corpus.

| type | rule |
|---|---|
| 1 wrong document | no frozen-relevant and no proposed-relevant document among the selected sources |
| 2 correct document, wrong passage | a relevant document is selected but none of its passages contains an evidence term (held-out questions, which have evidence terms) |
| 3 missing numeric / conditional evidence | the question asks for a quantity, date or condition and no selected passage has that shape |
| 4 fragmented context | half or more of the selected passages are under 100 characters |
| 5a / 5b outdated information | a selected passage states a deadline that has already passed (e.g. `2021.02.28.까지`) / the top source declares an edition > 3 years old |
| 6 conflicting sources | two selected documents state non-overlapping figures for the same unit |
| 7 cross-language failure | type 1, the question language differs from the expected document's, and the same scenario succeeds in another language |
| 8 incomplete benchmark labels | no frozen-relevant document selected, but a document proposed as relevant by the label review is |

In [ ]:
T = TR["corpus_v2"]; base = T[T.pipeline == "E2S"].set_index("question_id"); ev = T[T.pipeline == "E2S+EV"].set_index("question_id")
sq = assess_documents([dict(id=k, title=v["title"], text=v["content"]) for k, v in CORPUS.items()], SourceQualityConfig())
OLD_ED = {d["id"] for d in sq["documents"] if "old_declared_edition" in d["flags"]}
ok_frozen = {q: bool(set(base.loc[q, "documents"]) & FROZEN[q]) for q in base.index}
rows = []
for q, r in base.iterrows():
    rec = REC[q]; docs = set(r["documents"]); passages = [(s["document_id"], p) for s in r["sources"] for p in s["passages"]]
    e = ev.loc[q, "evidence"] or {}
    cats = []
    frozen_hit, ai_hit = bool(docs & FROZEN[q]), bool(docs & AI[q])
    if not frozen_hit and ai_hit: cats.append("8 incomplete labels")
    if not ai_hit:
        cats.append("1 wrong document")
        plang = MAN[rec["primary_document_id"]]["language"]
        sib = [k for k, x in REC.items() if x["scenario_id"] == rec["scenario_id"] and k != q]
        if rec["language"] != plang and any(ok_frozen.get(k) for k in sib): cats.append("7 cross-language")
    if ai_hit and DRAFT[q]["passage_support"]["evidence_terms"] and not any(supports(q, d, p) for d, p in passages if d in AI[q]):
        cats.append("2 right doc, wrong passage")
    if set(e.get("initially_missing", [])) & {"quantity", "date", "eligibility"}: cats.append("3 missing numeric/conditional")
    if passages and np.mean([p["chars"] < 100 for _, p in passages]) >= 0.5: cats.append("4 fragmented context")
    if any(stale_statement(p["text"]) for _, p in passages): cats.append("5a outdated: past deadline stated")
    if r["documents"] and r["documents"][0] in OLD_ED: cats.append("5b outdated: top source is an old edition")
    base_docs = {s["document_id"] for s in r["sources"]}
    if any(set(c["documents"]) <= base_docs for c in e.get("conflicts", [])): cats.append("6 conflicting sources")
    rows.append(dict(question_id=q, dataset=rec["dataset"], language=rec["language"], categories=cats or ["none"]))
AUD = pd.DataFrame(rows).explode("categories")
tab = AUD.groupby(["categories", "dataset"]).size().unstack(fill_value=0); tab["share of all 200"] = tab.sum(axis=1) / len(base)
display(tab.sort_index())
print("questions with at least one failure type:", (AUD.groupby("question_id").categories.apply(lambda s: "none" not in set(s))).sum(), "of", len(base))

In [ ]:
def example(cat, n=1):
    qs = AUD[AUD.categories == cat].question_id.tolist()[:n]
    for q in qs:
        r = base.loc[q]; p = next(((s["document_id"], x) for s in r["sources"] for x in s["passages"]), (None, {"text": ""}))
        e = ev.loc[q, "evidence"] or {}
        extra = " | top source declares a 2021 edition" if cat.startswith("5b") else ""
        if cat.startswith("6") and e.get("conflicts"): extra = f" | conflict: {e['conflicts'][0]}"
        if cat.startswith("5a"): extra = " | " + str(next((stale_statement(x["text"]) for s in r["sources"] for x in s["passages"] if stale_statement(x["text"])), None))
        display(Markdown(f"**{cat}** - `{q}` *{REC[q]['question']}*  \nexpected: {', '.join(TITLE.get(d, d)[:40] for d in FROZEN[q])}  \n"
                         f"top source: **{TITLE.get(p[0], p[0])[:50]}** (p. {p[1].get('page')}): {re.sub(chr(10), ' ', p[1]['text'])[:260]}{extra}"))
for cat in sorted(c for c in AUD.categories.unique() if c != "none"):
    example(cat)

## 2 · Document fragmentation (summary of notebook 13)

The guidebook fragmentation is an extraction defect (body font size estimated from the first 20 pages), which the heading-aware chunker turns into content loss. Full analysis, examples and statistics: `13_korean_pdf_chunk_quality.ipynb`.

In [ ]:
cs = pd.read_csv(CQ / "results/summary_by_variant.csv")
ch = {v: pd.read_csv(CQ / f"results/chunks_{v}.csv") for v in ("C0_existing", "C1_merge_short", "C2_paragraph", "C3_heading", "C4_table", "P1_prod_fixed_body")}
inv = json.loads((CQ / "investigation.json").read_text())
t = pd.DataFrame([dict(variant=v, chunks=len(d), median_chars=d.chars.median(), under_100=(d.chars < 100).mean(),
                       heldout_E2_MRR_frozen=cs[(cs.variant == v) & (cs.dataset == "heldout_v0.2") & (cs.method == "E2") & (cs.labels == "frozen")].rr.iloc[0],
                       heldout_E2_MRR_ai=cs[(cs.variant == v) & (cs.dataset == "heldout_v0.2") & (cs.method == "E2") & (cs.labels == "ai_sensitivity")].rr.iloc[0])
                  for v, d in ch.items()])
display(t.round(3))
print(f"production: {inv['font']['share_chars_tagged_heading']:.0%} of characters tagged as headings; "
      f"{inv['chunker']['share_boundaries_forced_by_heading']:.0%} of chunk boundaries forced by a heading change")

## 3 · Evidence completeness

The evidence layer (backend `app/services/assistant/evidence.py`, off by default) detects the information types a question asks for and checks whether the selected passages contain that *kind* of information; if not, it adds neighbouring chunks of cited documents and runs one bounded targeted search (<= 4 passages, <= 3,000 characters). Two kinds of measures:

* **heuristic sufficiency** - the layer's own check (circular: the same rule decides what to add, so it overstates the benefit);
* **independent measures** - (a) a supporting passage in the context according to the v0.3-draft passage criteria (evidence terms written with the benchmark, reviewed guidebook pages); (b) the share of the scenario's evidence terms found in the context. Both are computed on held-out questions.

In [ ]:
def context_metrics(T, labels=AI):
    out = []
    for _, r in T.iterrows():
        q = r.question_id; ps = [(s["document_id"], p) for s in r["sources"] for p in s["passages"]]
        terms = [t.lower() for t in DRAFT[q]["passage_support"]["evidence_terms"]]
        joined = " ".join((p["text"] or "").lower() for _, p in ps)
        e = r.evidence or {}
        out.append(dict(snapshot=r.snapshot, pipeline=r.pipeline, question_id=q, scenario_id=r.scenario_id, dataset=r.dataset,
                        language=r.language, needs=len(required_information(REC[q]["question"])) > 0,
                        support=float(any(supports(q, d, p, labels) for d, p in ps)) if has_criteria(q) else np.nan,
                        term_coverage=(sum(t in joined for t in terms) / len(terms)) if terms else np.nan,
                        sufficient=(float(not e.get("still_missing")) if e.get("needs") else np.nan),
                        initially_sufficient=(float(not e.get("initially_missing")) if e.get("needs") else np.nan),
                        added=len(e.get("added", [])), context_chars=sum(p["chars"] for _, p in ps), passages=len(ps),
                        conflicts=len(e.get("conflicts", [])), evidence_ms=r.evidence_ms or 0.0, select_ms=r.select_ms))
    return pd.DataFrame(out)
CM = pd.concat([context_metrics(T) for T in TR.values()], ignore_index=True)
held = CM[CM.dataset == "heldout_v0.2"]
g = held.groupby(["snapshot", "pipeline"]).agg(questions=("question_id", "count"), needs_detected=("needs", "mean"),
    heuristic_sufficient_before=("initially_sufficient", "mean"), heuristic_sufficient_after=("sufficient", "mean"),
    supporting_passage=("support", "mean"), evidence_term_coverage=("term_coverage", "mean"), passages_added=("added", "mean"),
    context_chars=("context_chars", "median"), potential_conflicts=("conflicts", "mean"), evidence_ms_median=("evidence_ms", "median"),
    evidence_ms_p90=("evidence_ms", lambda s: s.quantile(.9)))
display(g.round(3))
miss = collections.Counter(n for T in TR.values() for e in T[T.pipeline == "E2S+EV"].evidence if e for n in e["still_missing"])
need = collections.Counter(n for T in TR.values() for e in T[T.pipeline == "E2S+EV"].evidence if e for n in e["needs"])
display(pd.DataFrame([dict(type=k, questions_needing=need[k], still_missing_after_layer=miss[k]) for k in need]).set_index("type"))

In [ ]:
pairs = []
for snap in TR:
    d = CM[(CM.snapshot == snap) & (CM.dataset == "heldout_v0.2")]
    for a, b in (("E2S+EV", "E2S"), ("E4+EV", "E4"), ("E4", "E2S"), ("E4+EV", "E2S")):
        if {a, b} <= set(d.pipeline):
            x = d.rename(columns={"pipeline": "method"})
            for col in ("support", "term_coverage"):
                r = A.paired_difference(x.dropna(subset=[col]), a, b, col); r.update(snapshot=snap); pairs.append(r)
if len(TR) == 2:     # same questions, repaired guidebook (P1) vs production text: paired across corpora
    for pl in ("E2S", "E4"):
        x = CM[(CM.dataset == "heldout_v0.2") & (CM.pipeline == pl)].assign(method=lambda d: d.snapshot)
        for col in ("support", "term_coverage"):
            r = A.paired_difference(x.dropna(subset=[col]), "corpus_v2_p1", "corpus_v2", col)
            r.update(snapshot="P1 vs v2", a=f"{pl} [P1]", b=pl); pairs.append(r)
EVP = pd.DataFrame(pairs); EVP.to_csv(REP / "evidence_paired.csv", index=False)
display(EVP[["snapshot", "metric", "a", "b", "n", "mean_a", "mean_b", "diff", "lo", "hi", "a_better", "b_better"]].round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.4))
order = [p for p in ("E2S", "E2S+EV", "E4", "E4+EV", "E4bge", "E4bge+EV") if p in set(held.pipeline)]
snaps = list(TR)
for j, snap in enumerate(snaps):
    vals = [A.cluster_bootstrap(held[(held.snapshot == snap) & (held.pipeline == p)], "support") for p in order]
    y = np.arange(len(order)) + (j - (len(snaps) - 1) / 2) * 0.3
    ax.errorbar([v[0] for v in vals], y, xerr=[[v[0] - v[1] for v in vals], [v[2] - v[0] for v in vals]], fmt="o",
                color=(A.COLORS["E2"], A.COLORS["B0"])[j], capsize=3, markersize=7, linewidth=1.5,
                label={"corpus_v2": "corpus_v2 (production text)", "corpus_v2_p1": "guidebook re-extracted (P1)"}[snap])
    for v, yy in zip(vals, y): ax.text(v[0], yy - 0.16, f"{v[0]:.2f}", ha="center", fontsize=8, color=A.INK)
ax.set_yticks(np.arange(len(order)), order); ax.invert_yaxis(); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_xlabel("share of held-out questions with a supporting passage in the context (95% CI)")
ax.set_title("Supporting-passage retrieval (v0.3-draft criteria, pending review)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, frameon=False); fig.tight_layout()
print(A.save(fig, "14_supporting_passage", FIG))

## 4 · Source reliability (read-only checks; missing metadata = unknown)

From the corpus_v2 snapshot (document inventory + text). No metadata was changed. The validity and publication fields are not in the public snapshot for most documents and are therefore reported as unknown rather than assumed valid.

In [ ]:
invd = pd.read_csv(A.DATA / "snapshots/corpus_v2/document_inventory.csv")
alias = json.loads((A.DATA / "id_aliases.json").read_text())["aliases"]; back = {v: k for k, v in alias.items()}
docs = []
for r in invd.itertuples():
    pid = back.get(r.document_id, r.document_id)
    docs.append(dict(id=pid, title=r.title, authority_type=r.authority_type if isinstance(r.authority_type, str) else None,
                     source_url=r.source_url if isinstance(r.source_url, str) else None,
                     valid_until=r.valid_until if isinstance(r.valid_until, str) else None,
                     verification_status="approved" if r.current_status == "approved" else (r.current_status if isinstance(r.current_status, str) else None),
                     is_current=True, text=CORPUS.get(pid, {}).get("content")))
from urllib.parse import urlparse
SQ = assess_documents(docs, SourceQualityConfig(official_domains=["studyinkorea.go.kr", "hikorea.go.kr", "immigration.go.kr", "nhis.or.kr",
                                                                      "topik.go.kr", "sch.ac.kr", "lifeinkorea.go.kr", "korea.net"]))
sqd = pd.DataFrame(SQ["documents"])
display(sqd[["title", "authority", "domain", "validity", "published", "declared_edition_year", "version_status", "flags"]].assign(title=sqd.title.str[:38]))
display(pd.Series(SQ["unknown_counts"], name="documents with 'unknown'").to_frame())
display(pd.DataFrame(SQ["overlaps"]).assign(document=lambda d: d.document.map(lambda x: TITLE.get(x, x)[:35]),
                                            contained_in=lambda d: d.contained_in.map(lambda x: TITLE.get(x, x)[:35])))
conf = [(r.question_id, c) for T in TR.values() for r in T[T.pipeline == "E2S+EV"].itertuples() if r.evidence for c in r.evidence["conflicts"]]
print(f"potential conflicts flagged in E2S+EV contexts: {len(conf)} (question x unit); examples:")
for q, c in conf[:4]:
    print(" ", q, c["unit"], {TITLE.get(d, d)[:30]: v for d, v in c["values"].items()} if isinstance(c.get("values"), dict) else c)

## 5 · Benchmark: document relevance vs passage sufficiency (v0.3-draft)

The frozen v0.1/v0.2 files are unchanged (hash-checked by `benchmark_v03.py`). The draft adds (pending human review): proposed additional relevant documents, chunking-independent passage-support criteria (several passages can be correct) and the required information types. Development (pilot) and test (held-out) stay separate; nothing below was tuned on held-out results.

In [ ]:
D = pd.DataFrame(DRAFT.values())
display(D.groupby("split").agg(questions=("question_id", "count"),
        with_passage_criteria=("passage_support", lambda s: sum(bool(x["evidence_terms"] or x["guidebook_pages"]) for x in s)),
        with_proposed_documents=("proposed_additional_documents", lambda s: sum(bool(x) for x in s)),
        needs_information_type=("required_information", lambda s: sum(bool(x) for x in s))))
d = CM[(CM.snapshot == "corpus_v2") & (CM.pipeline == "E2S") & (CM.dataset == "heldout_v0.2")].copy()
d["doc_relevant"] = [bool(set(base.loc[q, "documents"]) & AI[q]) for q in d.question_id]
display(pd.crosstab(d.doc_relevant.map({True: "relevant document selected", False: "no relevant document"}),
                    d.support.map({1.0: "supporting passage present", 0.0: "no supporting passage"}), margins=True))

## 6 · Experiments: accuracy and cost

Document ranking (R@1, R@5, MRR, nDCG@5) for the baselines and reranking (Milestone 5 runs on corpus_v2) and for the context pipelines (documents in source order); held-out v0.2, frozen labels (official) and with the AI-proposed documents (sensitivity). Answer-level citation support and unsupported-answer rate require generated answers: **not evaluated** here (no LLM was called; no human-scored answers exist).

In [ ]:
exp = pd.concat([pd.read_csv(A.DATA / f"results/corpus_v2/experiments_{ds}.csv").assign(dataset=ds) for ds in ("pilot_v0.1", "heldout_v0.2")])
exp = exp[exp.method.isin(["B0", "E1", "E2", "E3_mminilm", "E3_bge-m3"])]   # E2S here = the assistant path from the traces[["method", "question_id", "scenario_id", "language", "dataset", "retrieved_documents", "total_ms"]]
ctx = []
for snap, T in TR.items():
    t = T[["pipeline", "question_id", "scenario_id", "language", "dataset", "documents", "select_ms", "evidence_ms"]].copy()
    t["method"] = (t.pipeline + ("" if snap == "corpus_v2" else " [P1]"))
    t["retrieved_documents"] = t.documents.map(json.dumps); t["total_ms"] = t.select_ms + t.evidence_ms.fillna(0)
    ctx.append(t.drop(columns=["pipeline", "documents", "select_ms", "evidence_ms"]))
ALL = pd.concat([exp] + ctx, ignore_index=True)
res = {}
for lab, L in (("frozen", FROZEN), ("ai_sensitivity", AI)):
    s = A.score_rankings(ALL, L)
    res[lab] = s
    t = A.summarize(s[s.dataset == "heldout_v0.2"], ["method"], ("r1", "r5", "rr", "ndcg5"))
    t["median_ms"] = t.method.map(s[s.dataset == "heldout_v0.2"].groupby("method").total_ms.median())
    display(Markdown(f"**Held-out v0.2 - {lab} labels**")); display(t.set_index("method").round(3))
SUMM = pd.concat([A.summarize(s[s.dataset == "heldout_v0.2"], ["method"], ("r1", "r5", "rr", "ndcg5")).assign(labels=l) for l, s in res.items()])
SUMM.to_csv(REP / "doc_ranking_summary.csv", index=False)

In [ ]:
PAIRS = [("E2", "B0"), ("E2", "E1"), ("E3_mminilm", "E2"), ("E3_bge-m3", "E2"), ("E2S+EV", "E2S"), ("E4", "E2S"),
         ("E4+EV", "E2S"), ("E2S [P1]", "E2S"), ("E4 [P1]", "E2S")]
pp = []
for lab, s in res.items():
    d = s[s.dataset == "heldout_v0.2"]
    for a, b in PAIRS:
        if {a, b} <= set(d.method):
            for col in ("rr", "r1"):
                r = A.paired_difference(d, a, b, col); r.update(labels=lab); pp.append(r)
PP = pd.DataFrame(pp)
prim = PP[(PP.labels == "frozen") & (PP.metric == "rr")].copy()
prim["p_approx"] = [2 * (1 - norm.cdf(abs(r["diff"]) / max((r["hi"] - r["lo"]) / 3.92, 1e-9))) for _, r in prim.iterrows()]
prim = prim.sort_values("p_approx"); m = len(prim); okk = True; sup = []
for i, (_, r) in enumerate(prim.iterrows()):
    okk = okk and r["p_approx"] <= 0.05 / (m - i); sup.append(okk)
prim["supported_after_holm"] = sup
PP.to_csv(REP / "paired_comparisons.csv", index=False)
display(prim[["a", "b", "n", "mean_a", "mean_b", "diff", "lo", "hi", "p_approx", "supported_after_holm"]].round(4))
display(PP[(PP.labels == "ai_sensitivity") & (PP.metric == "rr")][["a", "b", "diff", "lo", "hi", "ci_excludes_zero"]].round(4))

In [ ]:
s = res["frozen"]; d = s[s.dataset == "heldout_v0.2"]
show = [m for m in ("B0", "E1", "E2", "E3_mminilm", "E3_bge-m3", "E2S", "E2S+EV", "E4", "E4+EV", "E2S [P1]", "E4 [P1]") if m in set(d.method)]
lang = A.summarize(d[d.method.isin(show)], ["method", "language"], ("rr",)).pivot(index="method", columns="language", values="rr").reindex(show)
display(Markdown("**MRR by query language (held-out, frozen labels; n = 20 per language)**")); display(lang.round(3))
fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), gridspec_kw={"width_ratios": [1.25, 1]})
ax = axes[0]
for lab, color in (("frozen", A.COLORS["E2"]), ("ai_sensitivity", A.COLORS["E2S"])):
    t = SUMM[SUMM.labels == lab].set_index("method").reindex(show)
    y = np.arange(len(show)) + (0.15 if lab == "ai_sensitivity" else -0.15)
    ax.errorbar(t.rr, y, xerr=[t.rr - t.rr_lo, t.rr_hi - t.rr], fmt="o", color=color, capsize=3, markersize=6, linewidth=1.4,
                label={"frozen": "frozen labels", "ai_sensitivity": "+ proposed labels (not reviewed)"}[lab])
ax.set_yticks(np.arange(len(show)), show); ax.invert_yaxis(); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_xlabel("MRR, held-out v0.2 (95% CI)"); ax.set_title("Document ranking")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2, frameon=False)
ax = axes[1]
lat = d.groupby("method").total_ms.median().reindex(show); t = SUMM[SUMM.labels == "frozen"].set_index("method").reindex(show)
pts = [m for m in show if not m.endswith("+EV")]       # +EV has the same document ranking as its base pipeline
lat, t = lat.reindex(pts), t.reindex(pts)
ax.scatter(lat, t.rr, s=55, color=A.COLORS["B0"], zorder=3)
for mth in pts:
    ax.annotate(mth, (lat[mth], t.rr[mth]), textcoords="offset points", xytext=(5, 4), fontsize=8, color=A.INK)
ax.set_xscale("log"); ax.grid(axis="both"); ax.set_xlabel("median ms per question (log; Apple M1, rerank on GPU)")
ax.set_ylabel("MRR (frozen labels)"); ax.set_title("Accuracy vs cost (+EV: same MRR as base)")
fig.tight_layout(); print(A.save(fig, "14_accuracy_vs_cost", FIG))

## 7 · Conclusions and recommendation

In [ ]:
def P(a, b, lab="frozen", col="rr"):
    r = PP[(PP.labels == lab) & (PP.a == a) & (PP.b == b) & (PP.metric == col)]
    if r.empty: return "not evaluated"
    r = r.iloc[0]; return f"{100*r['diff']:+.1f} pp [{100*r['lo']:+.1f}, {100*r['hi']:+.1f}]"
def E(a, b, col="support", snap="corpus_v2"):
    r = EVP[(EVP.snapshot == snap) & (EVP.a == a) & (EVP.b == b) & (EVP.metric == col)]
    if r.empty: return "not evaluated"
    r = r.iloc[0]; return f"{100*r['diff']:+.1f} pp [{100*r['lo']:+.1f}, {100*r['hi']:+.1f}]"
G = g.reset_index().set_index(["snapshot", "pipeline"])
lat = d.groupby("method").total_ms.median()
cnt = AUD[AUD.categories != "none"].groupby("categories").question_id.nunique().sort_values(ascending=False)
wrong = AUD[AUD.categories == "1 wrong document"].question_id.unique()
xl_share = np.mean([REC[q]["language"] != MAN[REC[q]["primary_document_id"]]["language"] for q in wrong]) if len(wrong) else float("nan")
base_xl = np.mean([REC[q]["language"] != MAN[REC[q]["primary_document_id"]]["language"] for q in base.index])
def sig(a, b, col="support", snap="corpus_v2"):
    r = EVP[(EVP.snapshot == snap) & (EVP.a == a) & (EVP.b == b) & (EVP.metric == col)]
    return (not r.empty) and bool(r.iloc[0]["ci_excludes_zero"]) and r.iloc[0]["diff"] > 0
ev_rows = [e for e in TR["corpus_v2"][TR["corpus_v2"].pipeline == "E2S+EV"].evidence if e]
added_q = sum(1 for e in ev_rows if e["added"])
init_missing = sum(1 for e in ev_rows if e["initially_missing"]); still = sum(1 for e in ev_rows if e["still_missing"])
p1 = "corpus_v2_p1" in TR
ev_gain = sig("E2S+EV", "E2S") or sig("E2S+EV", "E2S", "term_coverage")
rr_gain = PP[(PP.labels == "frozen") & (PP.a == "E3_mminilm") & (PP.b == "E2") & (PP.metric == "rr")].iloc[0]
text = f"""### Findings

* **Failure audit (corpus_v2, assistant path E2S, 200 questions; multi-label, automatic rules).**
  {'; '.join(f'{k}: {v}' for k, v in cnt.items())}. {AUD[AUD.categories == 'none'].question_id.nunique()} questions match no failure rule.
  {xl_share:.0%} of wrong-document failures are cross-language questions (base rate {base_xl:.0%}).
* **Fragmentation.** An extraction defect (notebook 13): only 24 % of the guidebook's words reach chunk text in
  production; repaired variants keep 97-100 % but score lower against the frozen labels.
* **Evidence layer** (top-source check, development-calibrated): needed information types were missing in the top
  source for {init_missing} of {len(ev_rows)} questions (both sets); the layer added passages for {added_q} and
  {still} remain incomplete (recorded, not guessed). On held-out: supporting passage in context
  E2S {G.loc[('corpus_v2', 'E2S'), 'supporting_passage']:.2f} -> {G.loc[('corpus_v2', 'E2S+EV'), 'supporting_passage']:.2f}
  (paired {E('E2S+EV', 'E2S')}), evidence-term coverage {E('E2S+EV', 'E2S', 'term_coverage')}; median added cost
  {G.loc[('corpus_v2', 'E2S+EV'), 'evidence_ms_median']:.0f} ms (p90 {G.loc[('corpus_v2', 'E2S+EV'), 'evidence_ms_p90']:.0f} ms);
  document ranking {P('E2S+EV', 'E2S')} MRR. {'The improvement in evidence completeness is supported (CI excludes 0).' if ev_gain else 'No statistically supported improvement in the independent evidence measures.'}
* **Repaired guidebook (P1) vs production text, same questions:** supporting passage (E2S) {E('E2S [P1]', 'E2S', 'support', 'P1 vs v2')},
  evidence-term coverage {E('E2S [P1]', 'E2S', 'term_coverage', 'P1 vs v2')} - while frozen-label MRR falls
  ({P('E2S [P1]', 'E2S')}): the repair brings more answer-bearing text into the context than the frozen labels can credit.
* **Reranking.** E3 mMiniLM vs E2 {P('E3_mminilm', 'E2')}, E3 bge-m3 vs E2 {P('E3_bge-m3', 'E2')} MRR (frozen labels);
  with the assistant's selection: E4 vs E2S {P('E4', 'E2S')}; supporting passage E4 vs E2S {E('E4', 'E2S')}.
  Median cost per question {lat.get('E2S', float('nan')):.0f} ms (E2S) vs {lat.get('E4', float('nan')):.0f} ms (E4, mMiniLM on the M1 GPU;
  about 1.1 s on the M1 CPU).
* **Source reliability.** Unknown fields across {len(SQ['documents'])} documents: {', '.join(f'{k} {v}' for k, v in SQ['unknown_counts'].items())};
  overlaps: {'; '.join(f"{TITLE.get(o['document'], o['document'])[:30]} in {TITLE.get(o['contained_in'], o['contained_in'])[:30]} ({o['containment']:.0%})" for o in SQ['overlaps']) or 'none'};
  past-dated statements in held-out/pilot contexts: {AUD[AUD.categories.str.startswith('5a')].question_id.nunique()} questions.
* **Not evaluated:** citation support and unsupported-answer rate of generated answers (no LLM calls, no human-scored answers).
  {'The repaired-guidebook corpus (P1) is included above.' if p1 else 'The repaired-guidebook (P1) traces were not available when this notebook ran.'}

### Recommendation (none deployed)

1. **Fix PDF extraction** and re-extract the guidebook as a new version through approval (data integrity).
2. **Evidence-sufficiency layer:** {'enable after a shadow period' if ev_gain else 'keep off; run it in shadow mode (log `evidence_check` reports) - it detects incomplete evidence and records it, but the measured completeness gain is not established'}.
3. **Reranking (mMiniLM):** largest supported gain ({100*rr_gain['diff']:+.1f} pp MRR vs E2); needs ~1 GB RAM and ~1 s/question
   on CPU - only with a larger instance or a quantised model, measured on that hardware first.
4. **Source-quality report** for editors (unknowns, old editions, past deadlines, overlaps) - no automatic changes.
5. **Human review** of the v0.3 draft before any accuracy claim."""
(REP / "reliability_results.md").write_text("# Retrieval reliability (notebook 14)\n\n" + text + "\n")
display(Markdown(text))